# Sensor space analysis: Time domain analysis (ERP/ERF) in MNE-Python

`
Authors:
Marijn van Vliet, Britta Westner, Alexandre Gramfort, Denis A. Engemann
`

In [49]:
# This makes figures appear inside the notebook instead of in a new window.
%matplotlib qt

# We set the log-level of MNE-Python to 'warning' so the output is less verbose:
import mne
mne.set_log_level('error')

# Disable all warnings so they don't end up in the HTML.
import warnings
warnings.filterwarnings("ignore")

# Specify rendering backends
mne.viz.set_browser_backend("qt")

'qt'

## Setup

We already know this part from last time:
 

In [3]:
# These are the Python packages we will use in this session.
import numpy as np  # for numerical computations
import matplotlib.pyplot as plt  # for making custom figures
import mne  # for MEG and EEG data analysis

You should have downloaded the `ds000117-practical` folder. We have to let Python know, where to find this folder on your disk. You will have to adjust the path below to reflect your computer and path structure!
You can print the whole path and check the directory to double check it's correct.

In [4]:
# Change the following path to where the folder ds000117 is on your disk.
data_path = "./ds000117_pruned"  # `./` means the folder of this notebook

# Place where we want to save the processed data.
processed_path = "./processed"

# Change this to where you saved the epochs during the previous session.
epochs_fname = f"{processed_path}/sub-01_epo.fif"
print(epochs_fname)

./processed/sub-01_epo.fif


Now we can load the epochs we created during [the previous session](1-preprocessing).

In [5]:
epochs = mne.read_epochs(epochs_fname)

We can look at the epochs:

In [6]:
epochs.plot();

qt.core.qobject.connect: QObject::connect(QStyleHints, QStyleHints): unique connections require a pointer to member function of a QObject subclass


## Average the epochs to get ERF/ERP 

Let's first crop the data to the -0.2 to 1 seconds interval, so we zoom in more on the experimental effect.

In [16]:
epochs.crop(-0.2, 1)

<EpochsFIF | 146 events (all good), -0.2 – 1 s (baseline -0.5 – 0 s (baseline period was cropped after baseline correction)), ~160.2 MiB, data loaded,
 'face/famous/first': 25
 'face/famous/immediate': 10
 'face/famous/long': 14
 'face/unfamiliar/first': 25
 'face/unfamiliar/immediate': 12
 'face/unfamiliar/long': 10
 'scrambled/first': 25
 'scrambled/immediate': 14
 'scrambled/long': 11>

In [17]:
evoked = epochs.average()  # average across epochs to create the evoked response

In [18]:
evoked

<Evoked | '0.17 × face/famous/first + 0.07 × face/famous/immediate + 0.10 × face/famous/long + 0.17 × face/unfamiliar/first + 0.08 × face/unfamiliar/immediate + 0.07 × face/unfamiliar/long + 0.17 × scrambled/first + 0.10 × scrambled/immediate + 0.08 × scrambled/long' (average, N=146), -0.2 – 1 s, baseline -0.5 – 0 s (baseline period was cropped after baseline correction), 376 ch, ~8.0 MiB>

Let's look at our evoked field and evoked potential!

In [19]:
evoked.plot();

### Explore the different conditions in the data

We have seen in the last session, that the data has different events, which mark which stimulus was shown to the participants. The event/trigger structure is as follows:
- 5, 6, 7: famous faces
- 13, 14, 15: unfamiliar faces
- 17, 18, 19: scrambled faces

We have epoched the data according to these events previously. Let's see how we can use these event codes to work with the conditions!

We first look at how we registered the events:

In [5]:
epochs

<EpochsFIF | 146 events (all good), -0.5 – 2 s (baseline -0.5 – 0 s), ~325.7 MiB, data loaded,
 'face/famous/first': 25
 'face/famous/immediate': 10
 'face/famous/long': 14
 'face/unfamiliar/first': 25
 'face/unfamiliar/immediate': 12
 'face/unfamiliar/long': 10
 'scrambled/first': 25
 'scrambled/immediate': 14
 'scrambled/long': 11>

### Separate the conditions

Note that we used `/` separators in the event descriptions. These are inspired by file paths. For example, we can get all the epochs corresponding to "face" events:

In [10]:
epochs["face"]

<EpochsFIF | 96 events (all good), -0.5 – 2 s (baseline -0.5 – 0 s), ~216.6 MiB, data loaded,
 'face/famous/first': 25
 'face/famous/immediate': 10
 'face/famous/long': 14
 'face/unfamiliar/first': 25
 'face/unfamiliar/immediate': 12
 'face/unfamiliar/long': 10>

But also all the epochs where the stimulus was shown for a `long` duration:

In [9]:
epochs["long"]

<EpochsFIF | 35 events (all good), -0.5 – 2 s (baseline -0.5 – 0 s), ~83.4 MiB, data loaded,
 'face/famous/long': 14
 'face/unfamiliar/long': 10
 'scrambled/long': 11>

We can also select multiple events at the same time:

In [7]:
epochs["scrambled", "famous"]

<EpochsFIF | 99 events (all good), -0.5 – 2 s (baseline -0.5 – 0 s), ~223.1 MiB, data loaded,
 'face/famous/first': 25
 'face/famous/immediate': 10
 'face/famous/long': 14
 'scrambled/first': 25
 'scrambled/immediate': 14
 'scrambled/long': 11>

This functionality is very useful for creating contrasts between conditions. A common thing to do is to compute the evoked field in response to different conditions and compare them. The main conditions in this dataset are `famous` versus `unfamiliar` versus `scrambled`. Let's create `Evoked` objects for each of them:

In [20]:
evoked_famous = epochs["famous"].average()
evoked_unfamiliar = epochs["unfamiliar"].average()
evoked_scrambled = epochs["scrambled"].average()

print(evoked_famous)

<Evoked | '0.51 × face/famous/first + 0.20 × face/famous/immediate + 0.29 × face/famous/long' (average, N=49), -0.5 – 2 s, baseline -0.5 – 0 s, 376 ch, ~9.1 MiB>


From the output we see that the `comment` field of each `Evoked` object was automatically generated from all the different event types that went into the average. This is very verbose in our case. Let's set the `evoked.comment` field to something more sensible:

In [21]:
evoked_famous.comment = "famous faces"
evoked_unfamiliar.comment = "unfamiliar faces"
evoked_scrambled.comment = "scrambled stimuli"

print(evoked_famous)

<Evoked | 'famous faces' (average, N=49), -0.5 – 2 s, baseline -0.5 – 0 s, 376 ch, ~9.1 MiB>


That's better. Now to visually compare the evokeds. There are many functions for this, but the most obvious one to use is [`mne.viz.plot_compare_evokeds`](https://mne.tools/stable/generated/mne.viz.plot_compare_evokeds.html):

In [26]:
# Show the evoked for EEG channel 65 (in EEG we call the evoked the ERP)
mne.viz.plot_compare_evokeds([evoked_famous, evoked_unfamiliar, evoked_scrambled], picks="EEG065");

It is also possible to view all channels of a certain type in one big figure:

In [28]:
mne.viz.plot_compare_evokeds([evoked_famous, evoked_unfamiliar, evoked_scrambled], picks="mag", axes="topo");

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
      <li>Try clicking on the plot, what happens?</li>
      <li>Can you make the same plots for gradiometers and EEG sensors?</li>
    </ul>
</div>


MNE-Python also has some definitions for groups of channels that may be useful. For example, for the MegIn VectorView machine which recorded this dataset, it has the following channel groups:

- Vertex
- Left-temporal
- Right-temporal
- Left-occipital
- Right-occipital
- Left-frnmtal
- Right-frontal
  
We can load these with [`mne.channels.read_vectorview_selection`](https://mne.tools/stable/generated/mne.read_vectorview_selection.html) and then use them in the `picks` argument:

In [50]:
channel_group = mne.channels.read_vectorview_selection("Left-occipital", info=epochs.info)
mne.viz.plot_compare_evokeds([evoked_famous, evoked_unfamiliar, evoked_scrambled], picks=channel_group, show_sensors=True);

## Plot topographic maps

To get a sense of roughly where in the brain we can see a difference between the conditions, we can create topographic maps ("topomaps"), which we have seen before during the ICA part.

Next, we want to plot the topographies at the peak. We want to plot this for all channel types _and_ all conditions. To not wind up with 9 individual plots, let's see how to combine plots into one figure!

In [51]:
mne.viz.plot_evoked_topomap(evoked_famous, ch_type="mag", times=[0.137, 0.3, 0.6]);

It would be more useful to show the *difference* between two conditions. We can use the [`mne.combine_evoked`](https://mne.tools/stable/generated/mne.combine_evoked.html) function to do this:

In [62]:
famous_vs_scrambled = mne.combine_evoked([evoked_famous, evoked_scrambled], weights=[1, -1])
mne.viz.plot_evoked_topomap(famous_vs_scrambled, ch_type="mag", times=[0.137, 0.3, 0.6]);

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
      <li>Can you plot the difference between famous and unfamiliar faces?</li>
    </ul>
</div>


Another quick way to create topographic maps is to use the interactive capabilities of `mne.viz.plot_evoked`:

In [56]:
mne.viz.plot_evoked(famous_vs_scrambled);

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
      <li>Try clicking and dragging to select a time region, what happens?</li>
    </ul>
</div>


## Write evoked data to disk

As many MNE-Python objects, the `Evoked` object has a `.save` method:

In [57]:
evoked_famous.save(f"{processed_path}/sub-01_famous_ave.fif", overwrite=True) 

You can also write multiple conditions into one file:

In [58]:
mne.write_evokeds(f"{processed_path}/sub-01_ave.fif", [evoked_famous, evoked_unfamiliar, evoked_scrambled], overwrite=True)

### Reading evoked from disk

You can read the data back from the stored `.fif` file:

In [59]:
evokeds_list = mne.read_evokeds(f"{processed_path}/sub-01_ave.fif")

Or give the explicit name of the averaged condition when reading in the list:

In [61]:
evoked_famous = mne.read_evokeds(f"{processed_path}/sub-01_ave.fif", condition="famous faces")
evoked_famous

<Evoked | 'famous faces' (average, N=49), -0.5 – 2 s, baseline -0.5 – 0 s, 376 ch, ~9.1 MiB>

## Plot a 3D field map

We can also plot the topography in 3D. Since MNE-Python also plots the head surface here, we need a `-trans.fif` file, that tells us the _transform_ between the MEG device coordinate system and the head surface coordinate system.

In [70]:
# get the path of the transform file
# usually, this needs to be created by manual coregistration, but we have it all ready!
extra_path = "./extra_data_mne/"
trans_fname = f"{extra_path}/source_recon/sub-01/sub-01-trans.fif"

# we also need the Freesurfer subjects direcory:
subjects_dir = f"{extra_path}/freesurfer"

In [74]:
# first make the map
maps = mne.make_field_map(evoked_famous, trans=None, ch_type="meg")

# Plot it
evoked_famous.plot_field(maps, time=0.1);

In [71]:
mne.make_field_map?

Signature:
mne.make_field_map(
    evoked,
    trans='auto',
    subject=None,
    subjects_dir=None,
    ch_type=None,
    *,
    mode='fast',
    meg_surf='helmet',
    origin='auto',
    n_jobs=None,
    upsampling=1,
    head_source=('bem', 'head'),
    verbose=None,
)
Docstring:
Compute surface maps used for field display in 3D.

Parameters
----------
evoked : Evoked | Epochs | Raw
    The measurement file. Need to have info attribute.

trans : path-like | dict | instance of Transform | ``"fsaverage"`` | None
    If str, the path to the head<->MRI transform ``*-trans.fif`` file produced
    during coregistration. Can also be ``'fsaverage'`` to use the built-in
    fsaverage transformation.
    If trans is None, an identity matrix is assumed. ``"auto"`` (default) will load trans from the FreeSurfer
    directory specified by ``subject`` and ``subjects_dir`` parameters.

    .. versionchanged:: 0.19
        Support for ``'fsaverage'`` argument.
subject : str | None
    The subject n